In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
# Load main experiment dataset

mhs = pd.read_csv("../data/processed/mhs_main_experiment_annotations.csv")

print("Loaded shape:", mhs.shape)

Loaded shape: (49433, 52)


In [3]:
# Create comment-level table

comment_level = (
    mhs.groupby("comment_id")
    .agg(
        text_clean=("text_clean", "first"),
        target_type=("target_type", "first"),
        mean_hatespeech=("hatespeech", "mean")
    )
    .reset_index()
)

print("Comment-level shape:", comment_level.shape)
comment_level.head()

Comment-level shape: (19761, 4)


,comment_id,text_clean,target_type,mean_hatespeech
0,6,First off you look cool as fuck! Anyway if we ...,women_only,0.000000
1,7,\*points to posters asking for palestinian rig...,immigrant_only,1.000000
2,10,"They'll come back in your plan, also. Plus we ...",immigrant_only,1.333333
3,11,"eat my fuck, bitch",women_only,1.500000
4,12,She ugly anyways,women_only,1.000000


In [6]:
# Create hate severity buckets

comment_level["hatespeech_bucket"] = pd.cut(
    comment_level["mean_hatespeech"],
    bins=[-0.01, 0.5, 1.5, 2.0],
    labels=["low", "medium", "high"],
    include_lowest=True
)

comment_level["hatespeech_bucket"].value_counts()

hatespeech_bucket
low       12233
medium     3995
high       3533
Name: count, dtype: int64

In [8]:
# Final stratification label

def final_stratify_label(row):

    if row["target_type"] == "women_and_immigrant":
        return "women_and_immigrant"

    return f"{row['target_type']}__{row['hatespeech_bucket']}"

comment_level["stratify_label"] = comment_level.apply(
    final_stratify_label,
    axis=1
)

comment_level["stratify_label"].value_counts()

stratify_label
women_only__low           6476
immigrant_only__low       5526
women_only__medium        2539
women_only__high          2359
immigrant_only__medium    1385
immigrant_only__high      1092
women_and_immigrant        384
Name: count, dtype: int64

In [9]:
# First split: train vs temp

train_comments, temp_comments = train_test_split(
    comment_level,
    test_size=0.30,
    random_state=42,
    stratify=comment_level["stratify_label"]
)

print("Train comments:", len(train_comments))
print("Temp comments:", len(temp_comments))

Train comments: 13832
Temp comments: 5929


In [10]:
# Second split: validation vs test

val_comments, test_comments = train_test_split(
    temp_comments,
    test_size=0.50,
    random_state=42,
    stratify=temp_comments["stratify_label"]
)

print("Validation comments:", len(val_comments))
print("Test comments:", len(test_comments))

Validation comments: 2964
Test comments: 2965


In [11]:
# Extract comment ID sets

train_comment_ids = set(train_comments["comment_id"])
val_comment_ids = set(val_comments["comment_id"])
test_comment_ids = set(test_comments["comment_id"])

print("Train IDs:", len(train_comment_ids))
print("Validation IDs:", len(val_comment_ids))
print("Test IDs:", len(test_comment_ids))

Train IDs: 13832
Validation IDs: 2964
Test IDs: 2965


In [13]:
# Sanity checks

print(
    "Train and Validation:",
    len(train_comment_ids.intersection(val_comment_ids))
)

print(
    "Train and Test:",
    len(train_comment_ids.intersection(test_comment_ids))
)

print(
    "Validation and Test:",
    len(val_comment_ids.intersection(test_comment_ids))
)

Train and Validation: 0
Train and Test: 0
Validation and Test: 0


In [14]:
# Assign split labels back to annotation-level dataset

def assign_split(comment_id):

    if comment_id in train_comment_ids:
        return "train"

    elif comment_id in val_comment_ids:
        return "validation"

    elif comment_id in test_comment_ids:
        return "test"

    return "unknown"

mhs["split"] = mhs["comment_id"].apply(assign_split)

mhs["split"].value_counts()

split
train         34495
test           7953
validation     6985
Name: count, dtype: int64

In [15]:
# Split summary

split_summary = (
    mhs.groupby("split")
    .agg(
        annotation_rows=("comment_id", "count"),
        unique_comments=("comment_id", "nunique"),
        unique_annotators=("annotator_id", "nunique")
    )
    .reset_index()
)

split_summary

,split,annotation_rows,unique_comments,unique_annotators
0,test,7953,2965,4915
1,train,34495,13832,7697
2,validation,6985,2964,4571


In [16]:
# Stratification balance check

strat_balance = (
    comment_level.groupby("stratify_label")
    .size()
    .reset_index(name="total_comments")
)

train_balance = (
    train_comments.groupby("stratify_label")
    .size()
    .reset_index(name="train_comments")
)

val_balance = (
    val_comments.groupby("stratify_label")
    .size()
    .reset_index(name="validation_comments")
)

test_balance = (
    test_comments.groupby("stratify_label")
    .size()
    .reset_index(name="test_comments")
)

balance_df = strat_balance.merge(train_balance, on="stratify_label")
balance_df = balance_df.merge(val_balance, on="stratify_label")
balance_df = balance_df.merge(test_balance, on="stratify_label")

balance_df

,stratify_label,total_comments,train_comments,validation_comments,test_comments
0,immigrant_only__high,1092,764,164,164
1,immigrant_only__low,5526,3868,829,829
2,immigrant_only__medium,1385,970,207,208
3,women_and_immigrant,384,269,58,57
4,women_only__high,2359,1651,354,354
5,women_only__low,6476,4533,971,972
6,women_only__medium,2539,1777,381,381


In [17]:
# Women annotator distribution across splits

women_split_balance = (
    mhs[mhs["is_women_targeted"] == 1]
    .groupby(["split", "annotator_gender_group"])
    .size()
    .reset_index(name="count")
)

women_split_balance

,split,annotator_gender_group,count
0,test,men,1715
1,test,non_binary,32
2,test,prefer_not_to_say,17
3,test,self_describe,4
4,test,women,2231
5,train,men,8677
6,train,non_binary,151
7,train,prefer_not_to_say,65
8,train,self_describe,23
9,train,women,11535


In [18]:
# Ideology distribution across splits

ideology_split_balance = (
    mhs[mhs["is_immigrant_targeted"] == 1]
    .groupby(["split", "annotator_ideology_group"])
    .size()
    .reset_index(name="count")
)

ideology_split_balance

,split,annotator_ideology_group,count
0,test,conservative,437
1,test,extremely_conservative,134
2,test,extremely_liberal,572
3,test,liberal,1052
4,test,neutral,649
5,test,no_opinion,97
6,test,slightly_conservative,496
7,test,slightly_liberal,627
8,test,unknown,1
9,train,conservative,1642


In [19]:
# Save final datasets

os.makedirs("../data/processed", exist_ok=True)

train_df = mhs[mhs["split"] == "train"].copy()
val_df = mhs[mhs["split"] == "validation"].copy()
test_df = mhs[mhs["split"] == "test"].copy()

train_df.to_csv("../data/processed/train_annotations.csv", index=False)
val_df.to_csv("../data/processed/validation_annotations.csv", index=False)
test_df.to_csv("../data/processed/test_annotations.csv", index=False)

print("Saved train/validation/test annotation datasets.")

Saved train/validation/test annotation datasets.


In [22]:
# Save split report

os.makedirs("../results/tables", exist_ok=True)

report_path = "../results/tables/train_val_test_split_summary.txt"

with open(report_path, "w", encoding="utf-8") as f:

    f.write("TRAIN / VALIDATION / TEST SPLIT SUMMARY\n")
    f.write("=" * 70 + "\n\n")

    f.write("1. SPLIT SUMMARY\n")
    f.write("-" * 70 + "\n")
    f.write(split_summary.to_string())
    f.write("\n\n")

    f.write("2. STRATIFICATION BALANCE\n")
    f.write("-" * 70 + "\n")
    f.write(balance_df.to_string())
    f.write("\n\n")

    f.write("3. WOMEN TARGETED: GENDER DISTRIBUTION ACROSS SPLITS\n")
    f.write("-" * 70 + "\n")
    f.write(women_split_balance.to_string())
    f.write("\n\n")

    f.write("4. IMMIGRANT TARGETED: IDEOLOGY DISTRIBUTION ACROSS SPLITS\n")
    f.write("-" * 70 + "\n")
    f.write(ideology_split_balance.to_string())
    f.write("\n\n")


print(f"Saved report to: {report_path}")

Saved report to: ../results/tables/train_val_test_split_summary.txt


In [21]:
# Final check

mhs[[
    "comment_id",
    "target_type",
    "annotator_gender_group",
    "annotator_ideology_group",
    "split"
]].head()

,comment_id,target_type,annotator_gender_group,annotator_ideology_group,split
0,47101,immigrant_only,men,slightly_conservative,validation
1,43625,immigrant_only,men,neutral,train
2,12538,women_only,women,neutral,train
3,40171,women_only,women,slightly_conservative,train
4,1006,women_only,women,extremely_liberal,train
